# Movie Agent — 완전한 에이전트 루프

사용자 질문 → 도구 선택 → **실제 API 호출** → 결과를 LLM에 반환 → 최종 답변까지 루프. 멀티턴 메모리 지원.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os, json, requests
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"
BASE = "https://nomad-movies-2.nomadcoders.workers.dev"

print("setup done")

setup done


## 1. 실제 API를 호출하는 함수들

목록형 응답(인기/유사 영화)은 상위 5개 + 필요한 필드만 추려서 토큰을 아낍니다.

In [2]:
def slim_movie(m):
    return {
        "id": m["id"],
        "title": m["title"],
        "release_date": m.get("release_date"),
        "vote_average": m.get("vote_average"),
        "overview": (m.get("overview") or "")[:150],
    }

def get_popular_movies():
    movies = requests.get(BASE + "/movies").json()
    return [slim_movie(m) for m in movies[:5]]

def get_movie_details(movie_id):
    m = requests.get(BASE + "/movies/" + str(movie_id)).json()
    return {
        "id": m["id"],
        "title": m["title"],
        "release_date": m.get("release_date"),
        "runtime": m.get("runtime"),
        "vote_average": m.get("vote_average"),
        "genres": [g["name"] for g in m.get("genres", [])],
        "overview": m.get("overview"),
    }

def get_similar_movies(movie_id):
    movies = requests.get(BASE + "/movies/" + str(movie_id) + "/similar").json()
    return [slim_movie(m) for m in movies[:5]]

# 도구 이름 -> 실제 파이썬 함수 매핑
FUNCTIONS = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

print("functions defined")

functions defined


## 2. OpenAI tools 스키마

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "지금 인기 있는 영화 목록을 가져온다. 사용자가 인기 영화, 요즘 뜨는 영화를 물을 때 사용.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID의 상세 정보(줄거리, 장르, 평점 등)를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "영화의 고유 ID 번호"},
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화 ID와 유사한 영화 목록을 가져온다. 비슷한 영화를 추천해달라고 할 때 사용.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "기준이 되는 영화의 고유 ID 번호"},
                },
                "required": ["movie_id"],
            },
        },
    },
]

print(len(tools), "tools defined")

3 tools defined


## 3. 에이전트 루프

핵심: `tool_calls`가 있으면 함수를 **실제로 실행**하고, 결과를 `role: \"tool\"` 메시지로 추가한 뒤 모델을 다시 호출한다. `tool_calls`가 없으면(=최종 답변) 루프 종료. `messages`는 셀 밖에 유지되므로 멀티턴 대화가 된다.

In [4]:
messages = [
    {
        "role": "system",
        "content": "너는 영화 전문가 에이전트다. 사용자의 영화 질문에 답할 때 필요하면 도구를 호출해 실제 데이터를 가져와서 답한다. 답변은 한국어로 한다.",
    },
]

def ask(user_message, max_turns=5):
    messages.append({"role": "user", "content": user_message})

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        message = response.choices[0].message

        # 최종 답변이면 루프 종료
        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            print("Agent:", message.content)
            return

        # 도구 호출이면: assistant 메시지를 기록하고, 각 도구를 실제 실행
        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": c.id,
                        "type": "function",
                        "function": {"name": c.function.name, "arguments": c.function.arguments},
                    }
                    for c in message.tool_calls
                ],
            }
        )
        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"Agent: [{name}({args}) 호출]")
            result = FUNCTIONS[name](**args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

    print("Agent: (최대 턴 수 초과로 중단)")

print("agent ready")

agent ready


## 4. 예시 상호작용 (멀티턴)

In [5]:
ask("지금 인기 있는 영화 알려줘")

Agent: [get_popular_movies({}) 호출]


Agent: 현재 인기 있는 영화 목록이에요! 🎬

---

### 1️⃣ **Obsession** (2026)
- ⭐ 평점: **7.9**
- 📝 한 줄: 신비한 소원 버드나무를 부러뜨리며 짝사랑을 이루려는 한 순정남의 이야기

### 2️⃣ **Peddi** (2026)
- ⭐ 평점: **6.5**
- 📝 1980년대 인도 시골 마을에서 한 주민이 스포츠를 통해 마을의 자존심을 지키는 이야기

### 3️⃣ **Hai Jawani Toh Ishq Hona Hai** (2026)
- ⭐ 평점: **5.5**
- 📝 결혼 생활에서 충돌이 생긴 남자가 해외에서 새로운 로맨스를 찾는 이야기

### 4️⃣ **The Unknown Man** (2021)
- ⭐ 평점: **8.2**
- 📝 플랑드르 작가 루이가 남프랑스에서 영감을 찾는 이야기

### 5️⃣ **Mortal Kombat II** (2026)
- ⭐ 평점: **7.9**
- 📝 모탈 컴뱃의 챔피언들이 존 케이지와 함께 피할 수 없는 최후의 혈투를 벌이는 이야기

---

가장 높은 평점의 작품은 **The Unknown Man**(8.2점)이고, 최신 기대작으로는 **Mortal Kombat II**와 **Obsession**이 평점 7.9로 주목받고 있어요!

혹시 이 중에서 특정 영화의 더 자세한 정보나 비슷한 영화 추천이 필요하시면 말씀해주세요! 😊


In [6]:
ask("그중에 첫 번째 영화에 대해 더 알려줘")

Agent: [get_movie_details({'movie_id': 1339713}) 호출]


Agent: ## 🎬 Obsession (2026) 상세 정보

| 항목 | 내용 |
|------|------|
| **제목** | Obsession |
| **개봉일** | 2026년 5월 13일 |
| **러닝타임** | 108분 |
| **장르** | 🎃 **호러**, 😨 **스릴러** |
| **평점** | ⭐ **7.9 / 10** |

### 📖 줄거리
신비한 **"원 위시 윌로우(One Wish Willow)"** 나무를 꺾어 짝사랑하는 사람의 마음을 얻으려는 한 순정남(hopeless romantic)의 이야기입니다. 그는 정확히 자신이 원했던 것을 얻게 되지만, 곧 어떤 욕망은 **어둡고 불길한 대가**를 치르게 된다는 사실을 깨닫게 됩니다... 👻

즉, 소원을 이루는 대신 무서운 대가를 치러야 하는 전형적인 **소원 빌기 호러** 스토리로 보이네요! 관심 있으시면 비슷한 영화도 추천해드릴까요? 😊


In [7]:
ask("그 영화랑 비슷한 영화 추천해줘")

Agent: [get_similar_movies({'movie_id': 1339713}) 호출]


Agent: ## 🔍 Obsession과 비슷한 영화 추천

### 1️⃣ **The Flood** (2028)
- ⭐ 평점: 정보 없음
- 🚀 우주 탐사선 Spiritus Mundi가 외계 행성에서 이상한 신호를 감지하는 SF/호러
- *미개봉 기대작*

### 2️⃣ **Gladys** (2028)
- ⭐ 평점: 정보 없음
- 🧙‍♀️ 2025년 영화 *Weapons*의 프리퀄, 기생 마녀 Aunt Gladys의 과거 이야기
- *호러 팬이라면 주목!*

### 3️⃣ **Star Trek: First Contact** (1996)
- ⭐ 평점: **7.3 / 10**
- 🛸 보그(Borg)와의 전투, 피카드 함장의 활약! (SF 명작)

### 4️⃣ **Rebecca** (1940) 🏆
- ⭐ 평점: **7.9 / 10** - 알프레드 히치콕 감독!
- 🏚️ 젊은 신부가 죽은 전처의 그림자 속에서 살아가는 심리 스릴러
- *Obsession처럼 집착과 어두운 비밀을 다루는 클래식*

### 5️⃣ **The Hollow Child** (2018)
- ⭐ 평점: **6.3 / 10**
- 👻 위탁 가정에서 살아가는 소녀가 겪는 미스터리한 사건들

---

**추천 포인트!** 🎯
- **심리적 공포/집착**을 원한다면 → **Rebecca** (히치콕의 걸작!)
- **초자연적 호러**를 원한다면 → **The Hollow Child**
- **SF+호러 감성**을 원한다면 → **The Flood**

어떤 장르를 더 선호하시나요? 더 자세한 정보를 원하시면 말씀해주세요! 😊
